# POC Model Walkthrough

This notebook compares two versions of the same binary classifier:

- `with_2022_t1`: uses 2017 T1, 2017 T2, 2022 T1, and security features
- `without_2022_t1`: removes all 2022 T1 election features

Target:

- `1` if Macron wins the commune in 2022 second round
- `0` otherwise

In [20]:
# Define project directory paths for data access
from __future__ import annotations

import csv
import io
import json
import math
import random
from pathlib import Path

ROOT_DIR = Path.cwd().resolve().parents[1] if Path.cwd().name == 'ml' else Path.cwd().resolve()
while not (ROOT_DIR / 'data').exists() and ROOT_DIR != ROOT_DIR.parent:
    ROOT_DIR = ROOT_DIR.parent

ELECTION_DIR = ROOT_DIR / 'data' / 'gold' / 'election'
SECURITY_DIR = ROOT_DIR / 'data' / 'gold' / 'security'
OUTPUT_DIR = ROOT_DIR / 'src' / 'ml' / 'outputs'

print('ROOT_DIR =', ROOT_DIR)
print('ELECTION_DIR =', ELECTION_DIR)
print('SECURITY_DIR =', SECURITY_DIR)
print('OUTPUT_DIR =', OUTPUT_DIR)

ROOT_DIR = /Users/zainfrayha/Documents/EPSI/MSPR/electio-analytics-poc
ELECTION_DIR = /Users/zainfrayha/Documents/EPSI/MSPR/electio-analytics-poc/data/gold/election
SECURITY_DIR = /Users/zainfrayha/Documents/EPSI/MSPR/electio-analytics-poc/data/gold/security
OUTPUT_DIR = /Users/zainfrayha/Documents/EPSI/MSPR/electio-analytics-poc/src/ml/outputs


In [21]:
# Loop through collection of items and process each element
# Load and read data from data sources
def read_csv_rows(path: Path) -> list[dict[str, str]]:
    content = path.read_text(encoding='utf-8')
    return list(csv.DictReader(io.StringIO(content), delimiter=';'))


def write_csv_rows(path: Path, rows: list[dict[str, object]], fieldnames: list[str]) -> None:
    with path.open('w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def normalize_id_commune(value: str) -> str:
    parts = str(value).split('-', 1)
    if len(parts) == 2:
        dept = parts[0].strip().zfill(2)
        commune = parts[1].strip().replace('.0', '').zfill(3)
        return f'{dept}-{commune}'
    return str(value).strip()


def slugify(text: str) -> str:
    text = text.lower().strip()
    for old, new in [
        ('é', 'e'), ('è', 'e'), ('ê', 'e'), ('ë', 'e'),
        ('à', 'a'), ('â', 'a'), ('ä', 'a'),
        ('î', 'i'), ('ï', 'i'), ('ô', 'o'), ('ö', 'o'),
        ('ù', 'u'), ('û', 'u'), ('ü', 'u'), ('ç', 'c'),
        ("'", '_'), ('-', '_'), (' ', '_'), ('(', '_'), (')', '_'), ('/', '_')
    ]:
        text = text.replace(old, new)
    while '__' in text:
        text = text.replace('__', '_')
    return text.strip('_')


def to_float(value: str, default: float = 0.0) -> float:
    if value is None:
        return default
    raw = str(value).strip()
    if raw == '':
        return default
    try:
        return float(raw)
    except ValueError:
        return default


def to_int(value: str, default: int = 0) -> int:
    if value is None:
        return default
    raw = str(value).strip()
    if raw == '':
        return default
    try:
        return int(float(raw))
    except ValueError:
        return default


def sigmoid(x: float) -> float:
    x = max(-50.0, min(50.0, x))
    return 1.0 / (1.0 + math.exp(-x))


def dot_product(a: list[float], b: list[float]) -> float:
    return sum(x * y for x, y in zip(a, b))


def mean(values: list[float]) -> float:
    return sum(values) / len(values) if values else 0.0


def std(values: list[float], avg: float) -> float:
    if not values:
        return 1.0
    variance = sum((v - avg) ** 2 for v in values) / len(values)
    return math.sqrt(variance) or 1.0


print('Basic helper functions loaded.')

Basic helper functions loaded.


In [22]:
# Iterate through items and process each one
def classification_metrics(y_true: list[int], y_pred: list[int]) -> dict[str, float]:
    total = len(y_true)
    correct = sum(1 for yt, yp in zip(y_true, y_pred) if yt == yp)
    accuracy = correct / total if total else 0.0

    tp = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 1 and yp == 1)
    tn = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 0 and yp == 0)
    fp = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 0 and yp == 1)
    fn = sum(1 for yt, yp in zip(y_true, y_pred) if yt == 1 and yp == 0)

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    balanced_accuracy = (recall + specificity) / 2.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    return {
        'accuracy': accuracy,
        'balanced_accuracy': balanced_accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'tp': tp,
        'tn': tn,
        'fp': fp,
        'fn': fn,
    }


def stratified_split(rows: list[dict[str, object]], target_key: str, test_size: float = 0.2, seed: int = 42):
    random.seed(seed)
    grouped = {0: [], 1: []}
    for row in rows:
        grouped[int(row[target_key])].append(row)

    train_rows = []
    test_rows = []

    for _, group in grouped.items():
        random.shuffle(group)
        split_idx = max(1, int(round(len(group) * (1 - test_size))))
        train_rows.extend(group[:split_idx])
        test_rows.extend(group[split_idx:])

    random.shuffle(train_rows)
    random.shuffle(test_rows)
    return train_rows, test_rows


def prepare_matrices(rows: list[dict[str, object]], feature_names: list[str]):
    matrix = []
    for row in rows:
        matrix.append([to_float(row[feature], default=float('nan')) for feature in feature_names])
    return matrix


def impute_and_scale(train_matrix: list[list[float]], test_matrix: list[list[float]]):
    n_features = len(train_matrix[0])
    means = []
    stds = []

    for j in range(n_features):
        values = [row[j] for row in train_matrix if not math.isnan(row[j])]
        avg = mean(values)
        s = std(values, avg)
        means.append(avg)
        stds.append(s)

    def transform(matrix: list[list[float]]) -> list[list[float]]:
        transformed = []
        for row in matrix:
            new_row = []
            for j, value in enumerate(row):
                v = means[j] if math.isnan(value) else value
                new_row.append((v - means[j]) / stds[j])
            transformed.append(new_row)
        return transformed

    return transform(train_matrix), transform(test_matrix)


def fit_logistic_regression(X: list[list[float]], y: list[int], lr: float = 0.05, epochs: int = 3000, l2: float = 0.001):
    n_features = len(X[0])
    weights = [0.0] * n_features
    bias = 0.0
    n_samples = len(X)

    for epoch in range(epochs):
        grad_w = [0.0] * n_features
        grad_b = 0.0

        for row, target in zip(X, y):
            pred = sigmoid(dot_product(row, weights) + bias)
            error = pred - target
            for j in range(n_features):
                grad_w[j] += error * row[j]
            grad_b += error

        for j in range(n_features):
            grad_w[j] = grad_w[j] / n_samples + l2 * weights[j]
            weights[j] -= lr * grad_w[j]
        bias -= lr * (grad_b / n_samples)

        if epoch in [0, 499, 999, 1999, 2999]:
            print(f'Epoch {epoch + 1}/{epochs} finished')

    return weights, bias


def predict_classes(X: list[list[float]], weights: list[float], bias: float):
    probas = [sigmoid(dot_product(row, weights) + bias) for row in X]
    preds = [1 if p >= 0.5 else 0 for p in probas]
    return preds, probas


print('Training helper functions loaded.')

Training helper functions loaded.


In [23]:
# Load and read data from data sources
dim_election = read_csv_rows(ELECTION_DIR / 'dim_election.csv')
dim_candidat = read_csv_rows(ELECTION_DIR / 'dim_candidat.csv')
fact_resultats = read_csv_rows(ELECTION_DIR / 'fact_resultats.csv')
dim_indicateur = read_csv_rows(SECURITY_DIR / 'dim_indicateur_securite.csv')
fact_securite = read_csv_rows(SECURITY_DIR / 'fact_securite.csv')

print('dim_election rows =', len(dim_election))
print('dim_candidat rows =', len(dim_candidat))
print('fact_resultats rows =', len(fact_resultats))
print('dim_indicateur rows =', len(dim_indicateur))
print('fact_securite rows =', len(fact_securite))

print('Sample election row:', fact_resultats[0])
print('Sample security row:', fact_securite[0])

dim_election rows = 4
dim_candidat rows = 16
fact_resultats rows = 7378
dim_indicateur rows = 15
fact_securite rows = 37125
Sample election row: {'id_commune': '69-1', 'id_election': '2017_T2', 'id_candidat': 'CAND_001', 'inscrits': '257', 'abstentions': '39', 'votants': '218', 'blancs': '20', 'nuls': '7', 'exprimes': '191', 'pct_abs_ins': '15.18', 'pct_vot_ins': '84.82', 'pct_blancs_ins': '7.78', 'pct_blancs_vot': '9.17', 'pct_nuls_ins': '2.72', 'pct_nuls_vot': '3.21', 'pct_exprimes_ins': '74.32', 'pct_exprimes_vot': '87.61', 'voix': '93', 'pct_voix_ins': '36.19', 'pct_voix_exprimes': '48.69', 'date_integration_gold': '2026-03-24T19:55:58.226396'}
Sample security row: {'id_commune': '69-001', 'annee': '2016', 'id_indicateur_securite': 'SEC_001', 'code_departement': '69', 'code_commune': '001', 'code_insee_commune': '69001', 'nombre': '3.3925926', 'taux_pour_mille': '6.7806363', 'est_diffuse': 'ndiff', 'insee_pop': '358', 'insee_log': '163', 'complement_info_nombre': '3.3925926', 'comp

In [24]:
# Iterate through items and process each one
def build_election_dataset(include_2022_t1: bool = True):
    election_map = {row['id_election']: f"{row['annee_election']}_T{row['tour']}" for row in dim_election}
    candidate_map = {row['id_candidat']: {'nom': row['nom'], 'nuance': row['nuance']} for row in dim_candidat}

    features = {}
    target_scores = {}

    for row in fact_resultats:
        id_commune = normalize_id_commune(row['id_commune'])
        election_key = election_map[row['id_election']]
        candidate = candidate_map[row['id_candidat']]
        commune_features = features.setdefault(id_commune, {})

        if election_key == '2022_T2':
            score = to_float(row['voix'])
            best = target_scores.get(id_commune)
            if best is None or score > best[0]:
                target_scores[id_commune] = (score, candidate['nom'])
        else:
            if not include_2022_t1 and election_key == '2022_T1':
                continue
            for metric in ['inscrits', 'abstentions', 'votants', 'blancs', 'nuls', 'exprimes', 'pct_abs_ins', 'pct_vot_ins', 'pct_blancs_ins', 'pct_nuls_ins']:
                commune_features[f'{metric}_{election_key.lower()}'] = to_float(row[metric])
            commune_features[f"share_{slugify(candidate['nom'])}_{election_key.lower()}"] = to_float(row['pct_voix_exprimes'])

    targets = {id_commune: 1 if winner == 'MACRON' else 0 for id_commune, (_, winner) in target_scores.items()}
    return features, targets


def build_security_dataset():
    indicateur_map = {row['id_indicateur_securite']: slugify(row['indicateur']) for row in dim_indicateur}
    security_features = {}

    for row in fact_securite:
        year = to_int(row['annee'])
        if year not in (2021, 2022):
            continue
        id_commune = normalize_id_commune(row['id_commune'])
        commune_features = security_features.setdefault(id_commune, {})
        indicateur_slug = indicateur_map[row['id_indicateur_securite']]

        commune_features[f'sec_rate_{indicateur_slug}_{year}'] = to_float(row['taux_pour_mille'])
        commune_features[f'insee_pop_{year}'] = to_float(row['insee_pop'])
        commune_features[f'insee_log_{year}'] = to_float(row['insee_log'])

    return security_features


def run_experiment(experiment_name: str, include_2022_t1: bool):
    print('\n' + '=' * 80)
    print('EXPERIMENT =', experiment_name)
    print('include_2022_t1 =', include_2022_t1)
    print('=' * 80)

    election_features, targets = build_election_dataset(include_2022_t1=include_2022_t1)
    security_features = build_security_dataset()

    all_feature_names = set()
    dataset_rows = []

    for id_commune, target in targets.items():
        row = {'id_commune': id_commune, 'target_macron_wins_2022_t2': target}
        row.update(election_features.get(id_commune, {}))
        row.update(security_features.get(id_commune, {}))
        dataset_rows.append(row)
        all_feature_names.update(k for k in row.keys() if k not in {'id_commune', 'target_macron_wins_2022_t2'})

    feature_names = sorted(all_feature_names)
    for row in dataset_rows:
        for feature in feature_names:
            row.setdefault(feature, None)

    print('Dataset rows =', len(dataset_rows))
    print('Feature count =', len(feature_names))
    print('Class balance =', {0: sum(1 for r in dataset_rows if r['target_macron_wins_2022_t2'] == 0), 1: sum(1 for r in dataset_rows if r['target_macron_wins_2022_t2'] == 1)})
    print('First 10 features =', feature_names[:10])

    train_rows, test_rows = stratified_split(dataset_rows, 'target_macron_wins_2022_t2', test_size=0.2, seed=42)
    train_matrix = prepare_matrices(train_rows, feature_names)
    test_matrix = prepare_matrices(test_rows, feature_names)
    train_matrix, test_matrix = impute_and_scale(train_matrix, test_matrix)

    y_train = [int(row['target_macron_wins_2022_t2']) for row in train_rows]
    y_test = [int(row['target_macron_wins_2022_t2']) for row in test_rows]

    weights, bias = fit_logistic_regression(train_matrix, y_train)
    pred_classes, pred_probas = predict_classes(test_matrix, weights, bias)

    majority_class = round(sum(y_train) / len(y_train))
    baseline_pred = [majority_class] * len(y_test)

    baseline_metrics = classification_metrics(y_test, baseline_pred)
    model_metrics = classification_metrics(y_test, pred_classes)

    print('Baseline metrics =')
    print(json.dumps(baseline_metrics, indent=2))
    print('Model metrics =')
    print(json.dumps(model_metrics, indent=2))

    predictions = []
    for row, pred, proba in zip(test_rows, pred_classes, pred_probas):
        predictions.append({
            'id_commune': row['id_commune'],
            'target_macron_wins_2022_t2': row['target_macron_wins_2022_t2'],
            'predicted_class': pred,
            'predicted_proba_macron': proba,
        })

    feature_importance = []
    for feature, weight in sorted(zip(feature_names, weights), key=lambda item: abs(item[1]), reverse=True):
        feature_importance.append({
            'feature': feature,
            'coefficient': weight,
            'abs_coefficient': abs(weight),
        })

    experiment_dir = OUTPUT_DIR / experiment_name
    experiment_dir.mkdir(parents=True, exist_ok=True)

    write_csv_rows(experiment_dir / 'model_dataset.csv', dataset_rows, ['id_commune', 'target_macron_wins_2022_t2'] + feature_names)
    write_csv_rows(experiment_dir / 'test_predictions.csv', predictions, ['id_commune', 'target_macron_wins_2022_t2', 'predicted_class', 'predicted_proba_macron'])
    write_csv_rows(experiment_dir / 'feature_importance.csv', feature_importance, ['feature', 'coefficient', 'abs_coefficient'])

    metrics_payload = {
        'target_definition': '1 if MACRON wins the commune in 2022 second round, else 0',
        'baseline_metrics': baseline_metrics,
        'model_metrics': model_metrics,
        'dataset_rows': len(dataset_rows),
        'features_count': len(feature_names),
    }
    (experiment_dir / 'metrics.json').write_text(json.dumps(metrics_payload, indent=2), encoding='utf-8')

    print('Saved to =', experiment_dir)
    print('Top 5 features =')
    for row in feature_importance[:5]:
        print(row)

    return {
        'experiment_name': experiment_name,
        'include_2022_t1': include_2022_t1,
        'dataset_rows': len(dataset_rows),
        'features_count': len(feature_names),
        'baseline_metrics': baseline_metrics,
        'model_metrics': model_metrics,
        'predictions': predictions,
        'feature_importance': feature_importance,
    }


print('Experiment functions loaded.')

Experiment functions loaded.


In [25]:
result_with_2022_t1 = run_experiment('with_2022_t1', include_2022_t1=True)
result_without_2022_t1 = run_experiment('without_2022_t1', include_2022_t1=False)


EXPERIMENT = with_2022_t1
include_2022_t1 = True
Dataset rows = 267
Feature count = 89
Class balance = {0: 49, 1: 218}
First 10 features = ['abstentions_2017_t1', 'abstentions_2017_t2', 'abstentions_2022_t1', 'blancs_2017_t1', 'blancs_2017_t2', 'blancs_2022_t1', 'exprimes_2017_t1', 'exprimes_2017_t2', 'exprimes_2022_t1', 'inscrits_2017_t1']
Epoch 1/3000 finished
Epoch 500/3000 finished
Epoch 1000/3000 finished
Epoch 2000/3000 finished
Epoch 3000/3000 finished
Baseline metrics =
{
  "accuracy": 0.8148148148148148,
  "balanced_accuracy": 0.5,
  "precision": 0.8148148148148148,
  "recall": 1.0,
  "f1": 0.8979591836734693,
  "tp": 44,
  "tn": 0,
  "fp": 10,
  "fn": 0
}
Model metrics =
{
  "accuracy": 0.9259259259259259,
  "balanced_accuracy": 0.9159090909090909,
  "precision": 0.9761904761904762,
  "recall": 0.9318181818181818,
  "f1": 0.9534883720930233,
  "tp": 41,
  "tn": 9,
  "fp": 1,
  "fn": 3
}
Saved to = /Users/zainfrayha/Documents/EPSI/MSPR/electio-analytics-poc/src/ml/outputs/wit

In [26]:
# Output results
comparison = {
    'with_2022_t1': {
        'features_count': result_with_2022_t1['features_count'],
        'accuracy': result_with_2022_t1['model_metrics']['accuracy'],
        'balanced_accuracy': result_with_2022_t1['model_metrics']['balanced_accuracy'],
        'f1': result_with_2022_t1['model_metrics']['f1'],
    },
    'without_2022_t1': {
        'features_count': result_without_2022_t1['features_count'],
        'accuracy': result_without_2022_t1['model_metrics']['accuracy'],
        'balanced_accuracy': result_without_2022_t1['model_metrics']['balanced_accuracy'],
        'f1': result_without_2022_t1['model_metrics']['f1'],
    },
}

print('COMPARISON SUMMARY')
print(json.dumps(comparison, indent=2))

(OUTPUT_DIR / 'comparison.json').write_text(json.dumps(comparison, indent=2), encoding='utf-8')
print('Comparison file saved to =', OUTPUT_DIR / 'comparison.json')

COMPARISON SUMMARY
{
  "with_2022_t1": {
    "features_count": 89,
    "accuracy": 0.9259259259259259,
    "balanced_accuracy": 0.9159090909090909,
    "f1": 0.9534883720930233
  },
  "without_2022_t1": {
    "features_count": 67,
    "accuracy": 0.9074074074074074,
    "balanced_accuracy": 0.865909090909091,
    "f1": 0.942528735632184
  }
}
Comparison file saved to = /Users/zainfrayha/Documents/EPSI/MSPR/electio-analytics-poc/src/ml/outputs/comparison.json


In [27]:
# Loop through collection of items and process each element
print('TOP 10 FEATURES WITH 2022 T1')
for row in result_with_2022_t1['feature_importance'][:10]:
    print(row)

print('\nTOP 10 FEATURES WITHOUT 2022 T1')
for row in result_without_2022_t1['feature_importance'][:10]:
    print(row)

TOP 10 FEATURES WITH 2022 T1
{'feature': 'share_le_pen_2022_t1', 'coefficient': -1.429569246955142, 'abs_coefficient': 1.429569246955142}
{'feature': 'share_macron_2022_t1', 'coefficient': 1.4235039980651096, 'abs_coefficient': 1.4235039980651096}
{'feature': 'pct_blancs_ins_2017_t2', 'coefficient': -0.8051935480579349, 'abs_coefficient': 0.8051935480579349}
{'feature': 'sec_rate_vols_avec_armes_2021', 'coefficient': 0.7100470726284307, 'abs_coefficient': 0.7100470726284307}
{'feature': 'share_pecresse_2022_t1', 'coefficient': 0.659407239151568, 'abs_coefficient': 0.659407239151568}
{'feature': 'share_le_pen_2017_t2', 'coefficient': -0.6161271560261484, 'abs_coefficient': 0.6161271560261484}
{'feature': 'share_macron_2017_t2', 'coefficient': 0.6161271560261484, 'abs_coefficient': 0.6161271560261484}
{'feature': 'share_dupont_aignan_2017_t1', 'coefficient': 0.5769180776406844, 'abs_coefficient': 0.5769180776406844}
{'feature': 'sec_rate_violences_physiques_hors_cadre_familial_2022', 'co

## Interpretation

- If the model without `2022 T1` still performs well, the POC is stronger because it is less dependent on the immediately previous round.
- The main file to cite in your report is `comparison.json`.
- The detailed outputs for each experiment are stored in:
  - `src/ml/outputs/with_2022_t1`
  - `src/ml/outputs/without_2022_t1`